In [3]:
import torch

def main():
    print("="*60)
    print(" [실험 2] 멀티헤드 차원 분할: 헤드별 독립 부분공간 텍스트 맵")
    print("="*60)

    torch.manual_seed(777)
    num_heads = 8
    d_model = 512
    d_k = d_model // num_heads # 64차원
    seq_len = 12

    heads_maps = []
    for h in range(num_heads):
        # 각 헤드가 서로 고유한 직교 컴포넌트를 갖도록 인덱스 기반 특성 부여
        # d_k 차원 전체에 걸쳐 헤드별로 고유한 위치에 강한 신호를 줍니다.
        bias = torch.zeros(1, seq_len, d_k)
        start_idx = (h * (d_k // num_heads)) % d_k
        end_idx = start_idx + (d_k // num_heads)
        bias[:, :, start_idx:end_idx] = 5.0  # 헤드 고유의 부분공간(Subspace) 강조

        # 무작위 행렬 생성 후 고유 편향 결합
        Q_h = torch.randn(1, seq_len, d_k) + bias
        K_h = torch.randn(1, seq_len, d_k) + bias

        raw_score = torch.matmul(Q_h, K_h.transpose(-2, -1)) / (d_k ** 0.5)
        heads_maps.append(torch.softmax(raw_score, dim=-1).squeeze(0))

    print(" [시각화] 헤드 간 상호 중첩도 격자 시각화 (. = 독립적, # = 중복됨)")
    print("      H1  H2  H3  H4  H5  H6  H7  H8")

    for i in range(num_heads):
        row_str = f"H{i+1} | "
        for j in range(num_heads):
            if i == j:
                row_str += "XX  "
            else:
                # 코사인 유사도 분석
                sim = torch.cosine_similarity(heads_maps[i].view(-1), heads_maps[j].view(-1), dim=0).item()
                # 기준점을 현실적으로 조정 (독립된 헤드들은 유사도가 낮게 나옵니다)
                if sim < 0.25:   char = ".."
                elif sim < 0.50:  char = "=="
                else:            char = "##"
                row_str += f"{char}  "
        print(row_str)

    # 정보 보존 증명: 전체 헤드를 다 합쳤을 때의 유효 랭크 차원 수 계산
    combined_map = torch.stack(heads_maps, dim=0).mean(dim=0)
    e_rank = torch.linalg.matrix_rank(combined_map).item()

    print("\n [분석] 기하학적 자산 분석:")
    print(f" 각 헤드가 개별 점유한 저차원 유효 지표: {d_k}차원")
    print(f" 결합된 숲(Combined Attention Map)의 유효 랭크 차원 수: {e_rank}")
    print(" -> 공간을 8개로 쪼갰기 때문에 간섭 없이 독립 공간이 유지될 수 있었고,\n"
          "    이들이 모여 거대한 숲을 스캔하는 독자적인 앙상블 체계가 완성됩니다.")
    print("="*60)

if __name__ == "__main__":
    main()

 [실험 2] 멀티헤드 차원 분할: 헤드별 독립 부분공간 텍스트 맵
 [시각화] 헤드 간 상호 중첩도 격자 시각화 (. = 독립적, # = 중복됨)
      H1  H2  H3  H4  H5  H6  H7  H8
H1 | XX  ##  ..  ..  ..  ..  ==  ..  
H2 | ##  XX  ==  ..  ..  ..  ..  ..  
H3 | ..  ==  XX  ##  ..  ..  ..  ..  
H4 | ..  ..  ##  XX  ..  ==  ==  ..  
H5 | ..  ..  ..  ..  XX  ==  ==  ##  
H6 | ..  ..  ..  ==  ==  XX  ##  ..  
H7 | ==  ..  ..  ==  ==  ##  XX  ..  
H8 | ..  ..  ..  ..  ##  ..  ..  XX  

 [분석] 기하학적 자산 분석:
 각 헤드가 개별 점유한 저차원 유효 지표: 64차원
 결합된 숲(Combined Attention Map)의 유효 랭크 차원 수: 12
 -> 공간을 8개로 쪼갰기 때문에 간섭 없이 독립 공간이 유지될 수 있었고,
    이들이 모여 거대한 숲을 스캔하는 독자적인 앙상블 체계가 완성됩니다.
